Load the search yaml


In [ ]:
from nam.utils.config import load_search_config

config_set, search_space = load_search_config(r"C:\Users\maart\OneDrive\Documenten\Universiteit\Scriptie\python_repo\thesis-nam\configs\compas_search.yaml")

print(config_set)
print(search_space)





Load the data in Optuna

In [ ]:
import optuna
from nam.models.nam import NAM
from nam.data.data_utils import load_compas, preprocess, split
from torch.utils.data import DataLoader
from nam.data.dataset import NAMDataset
from nam.training.trainer import Trainer
from pathlib import Path

n_trials = 50 

# --- Data ---
df = load_compas(config_set['dataset_path'])
X, y, feature_names = preprocess(df)
X_train, X_val, X_test, y_train, y_val, y_test = split(
    X, y, config_set["val_frac"], config_set["test_frac"], config_set["seed"]
)

train_loader = DataLoader(NAMDataset(X_train, y_train), batch_size=config_set["batch_size"], shuffle=True)
val_loader   = DataLoader(NAMDataset(X_val,   y_val),   batch_size=config_set["batch_size"], shuffle=False)
test_loader  = DataLoader(NAMDataset(X_test,  y_test),  batch_size=config_set["batch_size"], shuffle=False)

def suggest_hyperparams(trial, search_space: dict) -> dict:
    params = {}
    for name, spec in search_space.items():
        if spec["type"] == "float_log":
            params[name] = trial.suggest_float(name, spec["low"], spec["high"], log=True)
        elif spec["type"] == "float":
            params[name] = trial.suggest_float(name, spec["low"], spec["high"])
        elif spec["type"] == "categorical":
            params[name] = trial.suggest_categorical(name, spec["choices"])
        elif spec["type"] == "int":
            params[name] = trial.suggest_int(name, spec["low"], spec["high"])
    return params


def objective(trial):
    trial_params = suggest_hyperparams(trial, search_space)
    
    model = NAM(
        num_features=X_train.shape[1],
        num_units=config_set["num_units"],
        hidden_sizes=config_set["hidden_sizes"],
        dropout=trial_params["dropout"],
        feature_dropout=trial_params["feature_dropout"],
        activation=trial_params["activation"],
    )

    trainer = Trainer(
        model=model,
        lr=trial_params["lr"],
        decay_rate=config_set["decay_rate"],
        output_regularization=trial_params["output_regularization"],
        l2_regularization=trial_params["l2_regularization"],
        task=config_set["task"],
        num_epochs=config_set["num_epochs"],
        patience=config_set["patience"],
        val_check_interval=config_set["val_check_interval"],
    )

    trainer.train(train_loader, val_loader, trial)
    return trainer.best_val_metric



direction = "maximize" if config_set["task"] == "classification" else "minimize"
dataset_name = Path(config_set["dataset_path"]).stem

storage = f"sqlite:///runs/{dataset_name}_optuna.db"
study_name = f"{dataset_name}_search"

study = optuna.create_study(
    study_name=study_name,
    storage=storage,
    load_if_exists=True,
    direction=direction
)
study.optimize(objective, n_trials=config_set["n_trials"])

print(study.best_trial.value)   # best metric achieved
print(study.best_trial.params)  # dict of best hyperparameters

Convert final output back to YAML

In [ ]:
import yaml

# Merge fixed params with best trial params
best_params = {**config_set, **study.best_trial.params}

# Remove tuning-only keys not part of NAMConfig
best_params.pop("n_trials", None)

output_path = Path(r"C:\Users\maart\OneDrive\Documenten\Universiteit\Scriptie\python_repo\thesis-nam\configs") / f"{dataset_name}_tuned.yaml"

with open(output_path, "w") as f:
    yaml.dump(best_params, f, default_flow_style=False, sort_keys=False)

print(f"Best config saved to {output_path}")